# ModAdd p=7, m=31 OPD Rollout Comparison

This notebook builds per-eta eval plots comparing `forward_kl_simple` and `reverse_kl_tm` on the Hydra-era ModAdd runs for `p=7, m=31`.

Each per-eta figure includes up to four curves:

- `Forward KL Simple, greedy rollout`
- `Forward KL Simple, sampled rollout`
- `Reverse KL TM, greedy rollout`
- `Reverse KL TM, sampled rollout`

The default configuration below targets `eta in {0.0, 0.1, 0.3, 0.5, 0.7, 0.9}` and uses the run-name patterns with either `...-t1p0-...` or `...-rollgreedy-studt1p0-...`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Resolve the repo root whether the notebook is opened from the repo root or the notebooks/ folder.
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.plot_modadd_opd_rollout_runs import (
    build_runs_df,
    default_output_dir,
    discover_modadd_opd_runs,
    plot_per_eta,
)

RUN_ROOT = ROOT

P = 7
M = 31
SUBSET_SIZE = 1_000_000
SEED = 20260417
ETAS = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9]
OBJECTIVES = ['forward_kl_simple', 'reverse_kl_tm']

SAVE_PLOTS = True
PLOT_EXPORT_DIR = default_output_dir(ROOT, p=P, m=M, subset_size=SUBSET_SIZE, seed=SEED)
PLOT_EXPORT_DIR


In [ ]:
# Discover the matching OPD runs and load their eval histories.
records = discover_modadd_opd_runs(
    RUN_ROOT,
    p=P,
    m=M,
    subset_size=SUBSET_SIZE,
    seed=SEED,
    etas=ETAS,
    objectives=OBJECTIVES,
)

if not records:
    raise RuntimeError(
        'No matching ModAdd OPD runs were found. Check RUN_ROOT / P / M / SUBSET_SIZE / SEED / ETAS against your out-dir names.'
    )

runs_df, run_data = build_runs_df(records)
runs_df


In [ ]:
# Coverage summary so we can confirm which of the four expected curves exist at each eta.
coverage_df = (
    pd.concat(run_data.values(), ignore_index=True)
    .groupby(['objective', 'rollout_label', 'eta'], as_index=False)
    .agg(
        n_points=('iter', 'size'),
        min_iter=('iter', 'min'),
        max_iter=('iter', 'max'),
        final_clean_full_exact=('val/clean_full_exact', 'last'),
        final_clean_final_exact=('val/clean_final_exact', 'last'),
    )
    .sort_values(['eta', 'objective', 'rollout_label'])
    .reset_index(drop=True)
)
coverage_df


In [ ]:
# Save one per-eta figure for clean_full_exact and also show it in the notebook.
plot_per_eta(
    run_data,
    runs_df,
    metric='val/clean_full_exact',
    out_dir=PLOT_EXPORT_DIR if SAVE_PLOTS else None,
    show=True,
)


In [ ]:
# Save one per-eta figure for clean_final_exact and also show it in the notebook.
plot_per_eta(
    run_data,
    runs_df,
    metric='val/clean_final_exact',
    out_dir=PLOT_EXPORT_DIR if SAVE_PLOTS else None,
    show=True,
)
